# Knowlytix GMS — post-install smoke notebook

Interactive companion to `knowlytix-smoke`. Exercises the four
Knowlytix wheels (`knowlytix-core`, `knowlytix-doc`, `knowlytix-bench`,
`knowlytix-harness`) against the shipped smoke fixtures. **Key-free** —
no provider API keys required; no GPU; no network.

**Prerequisite:** install the wheels first (see `docs/internal_team_install.md`).

Then open this notebook: `jupyter lab $(knowlytix-smoke --notebook-path)`

**Scope:** this notebook is a DEMO surface for humans. The canonical
gate for "did the install work?" is `knowlytix-smoke` — it runs
non-interactively in CI after every wheel build.

## 1. Imports resolve and `__all__` is declared

Each of the four shipped packages must declare an `__all__` listing
its public surface. Walking `__all__` + `hasattr` catches
the two common drift cases: a wheel that ships without a promised
symbol, and a wheel that promises a symbol it no longer imports.

In [ ]:
import importlib

PACKAGES = ("knowlytix.core", "knowlytix.knowledge", "knowlytix.benchmark", "knowlytix.harness.testing")
for name in PACKAGES:
    mod = importlib.import_module(name)
    assert hasattr(mod, "__all__") and mod.__all__, f"{name} missing __all__"
    for sym in mod.__all__:
        assert hasattr(mod, sym), f"{name}.{sym} not resolvable"
    print(f"{name}: {len(mod.__all__)} public symbols OK")

## 2. Settings classes instantiate with their defaults

If the `pydantic-settings` schema drifted (field renamed, required
field added without default, env_prefix changed) the wheels crash
here. Tweak env vars in-cell to confirm the override paths work.

In [ ]:
from knowlytix.core import GMSCoreSettings, LLMSettings, ModelPurpose
from knowlytix.knowledge import DocGMSSettings
from knowlytix.benchmark import FinStructBenchSettings
from knowlytix.harness.settings import GMSHSettings

core = GMSCoreSettings()
llm = LLMSettings()
doc = DocGMSSettings()
fsb = FinStructBenchSettings()
gmsh_settings = GMSHSettings()

# Try overriding a tuning knob via env var:
#   GMS_LLM_MODEL=ollama/llama3 python -m jupyter lab
print(f"LLM default model:          {llm.resolve(ModelPurpose.DEFAULT)!r}")
print(f"LLM judge model (if set):   {llm.resolve(ModelPurpose.JUDGE)!r}")
print(f"gms-core stores_dir:        {core.stores_dir}")
print(f"knowlytix-knowledge max_pages:           {doc.max_pages}")
print(f"knowlytix-benchmark float_tol:   {fsb.float_tol}")
print(f"gmsh max_workers:           {gmsh_settings.max_workers}")

## 3. Smoke fixtures reachable via `importlib.resources`

The README quickstarts in `packaging/{knowlytix-knowledge,knowlytix-benchmark,knowlytix-harness}/README.md`
all depend on the fixtures landing in the wheel and resolving via
`importlib.resources.files(...)`. The `__init__.py` stubs under each
fixture package make them discoverable in both regular and editable
installs.

In [ ]:
from importlib.resources import files

sample_md = (files("knowlytix.knowledge.fixtures.smoke") / "sample.md").read_text()
print(f"knowlytix-knowledge sample.md: {len(sample_md)} chars")
print(sample_md[:240] + "…\n")

import json

q = json.loads((files("knowlytix.benchmark.fixtures.smoke") / "questions.json").read_text())
p = json.loads((files("knowlytix.benchmark.fixtures.smoke") / "predictions.json").read_text())
print(f"knowlytix-benchmark: {len(q['questions'])} questions, {len(p['predictions'])} predictions")

doe = json.loads((files("knowlytix.harness.fixtures.smoke") / "doe_smoke.json").read_text())
gt = json.loads((files("knowlytix.harness.fixtures.smoke") / "ground_truth.json").read_text())
print(f"harness DOE: {len(doe['scenarios'])} scenarios, {len(doe['factors'])} factors")
print(f"harness ground truth: {len(gt['entities'])} entities, {len(gt['relations'])} relations, {len(gt['facts'])} facts")

## 4. `knowlytix.benchmark.score_answer` — the key-free entry point

`score_answer(answer, ground_truth)` is the graph-only path that works
without an LLM. Here we pair each shipped prediction with the matching
ground truth, coerce the type, and call the scorer — exercising the
float-tolerance, int, bool, and set parse branches.

In [ ]:
from knowlytix.benchmark import ScoreResult, score_answer

preds_by_id = {p["id"]: p["answer"] for p in p["predictions"]}

for question in q["questions"]:
    qid = question["id"]
    gt_value = question["ground_truth"]
    raw = preds_by_id.get(qid)
    answer = raw
    if isinstance(gt_value, bool):
        answer = str(raw).strip().lower() in ("true", "1", "yes")
    elif isinstance(gt_value, float):
        try:
            answer = float(raw)
        except (TypeError, ValueError):
            pass
    elif isinstance(gt_value, int):
        try:
            answer = int(raw)
        except (TypeError, ValueError):
            pass
    result = score_answer(answer, gt_value)
    assert isinstance(result, ScoreResult)
    status = "✓" if result.correct else "✗"
    print(f"  {status} [{qid}] {question['type']:<18} answer={answer!r:<24}  gt={gt_value!r}  → {result.detail}")


## 5. harness DOE fixtures — shape check

Running a full DOE campaign (`DOEGMSBenchmark`) needs an LLM client,
so we keep the notebook gate at the fixture-shape level: assert the
schema matches what the `gmsh` CLI expects to consume. Full
campaign walkthroughs live in the end-to-end demos under
`packaging/gmsh/README.md` §Programmatic quickstart.

In [ ]:
for key in ("campaign", "factors", "scenarios", "gates"):
    assert key in doe, f"doe_smoke.json missing '{key}'"
for key in ("entities", "relations", "facts"):
    assert key in gt, f"ground_truth.json missing '{key}'"

print("DOE campaign:", doe["campaign"])
print("Factors:", [f["name"] for f in doe["factors"]])
print("Scenarios:", [s["id"] for s in doe["scenarios"]])
print()
print("Ground-truth entities:", [e["id"] for e in gt["entities"]])
print("Ground-truth relations:", [r["predicate"] for r in gt["relations"]])

## Next steps

If everything above ran cleanly, the wheels are healthy. To keep
exploring:

* **knowlytix-doc** — feed your own markdown or PDF to `ingest_document()`;
  see `packaging/knowlytix-doc/README.md` for the full quickstart.
* **knowlytix-bench** — pick a bundled instance and run
  `Benchmark(...).run()`.
* **knowlytix-harness** — run `knowlytix-harness --help` or
  `knowlytix-doe --help` from a terminal to see the campaign CLI;
  the programmatic path uses `DOEGMSBenchmark` + `make_evaluator`.

For automated gating, keep running `knowlytix-smoke` — this
notebook is for humans; the CLI entry point is for CI.